In [55]:
import sys
import pandas as pd
from pathlib import Path
import re 
import numpy as np

sys.path.append(str(Path('..').resolve()))
from src.config import RAW_PATH, listar_tablas_raw, get_raw_file, get_silver_file

In [16]:
# ==========================================
# LISTAMOS LAS TABLAS EN RAW
# ==========================================

tablas = listar_tablas_raw()

print(f"📊 Procesando {len(tablas)} tablas...\n")

# Diccionario para almacenar todos los DataFrames transformados
dataframes = {}

📊 Procesando 20 tablas...



## Descripcion de datos

In [ ]:
# Describir cada tabla

for tabla in sorted(tablas):
    # Cargar datos
    df = pd.read_csv(get_raw_file(tabla))
    
    # Descripción básica
    print(f"📌 {tabla.upper()}".center(50, '-'))
    print(f"📏 Shape: {df.shape}")
    print(f"\n📋 Info:")
    df.info()


## Manejo de datos nulos y duplicados

In [18]:
# Función para contar nulos por columna
def cnt_nulos_columnas(dataframe, columnas=None):
    """
    Versión mejorada de tu función original
    """
    if columnas is None:
        columnas = dataframe.columns.tolist()
    
    dfResumenNulos = pd.DataFrame(
        {"Columna": [], "Cantidad_Nulos": [], "Porcentaje_Nulos (%)": []}
    )
    
    for columna in columnas:
        cnt_nulos = dataframe[columna].isnull().sum()
        porcentaje_nulos = (cnt_nulos / len(dataframe)) * 100
        dfResumenNulos.loc[len(dfResumenNulos)] = [
            columna, 
            cnt_nulos, 
            round(porcentaje_nulos, 2)
        ]
    
    return dfResumenNulos

In [19]:
# Describir cada tabla con nulos y duplicados
for tabla in sorted(tablas):
    df = pd.read_csv(get_raw_file(tabla))
    
    print(f"📌 {tabla.upper()}".center(50, '-'))
    
    # Lista de columnas
    lista_columnas = df.columns.tolist()
    print(f"\n📋 Columnas: {lista_columnas}")
    
    # Resumen de nulos
    print(f"\n📊 Resumen de nulos:")
    print(cnt_nulos_columnas(df).to_string(index=False))
    
    # Resumen de duplicados
    duplicados = df.duplicated().sum()
    print(f"\n📊 Filas duplicadas: {duplicados} ({round(duplicados/len(df)*100, 2)}%)")

-------------------📌 CUSTOMERS--------------------

📋 Columnas: ['id', 'company', 'last_name', 'first_name', 'email_address', 'job_title', 'business_phone', 'home_phone', 'mobile_phone', 'fax_number', 'address', 'city', 'state_province', 'zip_postal_code', 'country_region', 'web_page', 'notes', 'attachments']

📊 Resumen de nulos:
        Columna  Cantidad_Nulos  Porcentaje_Nulos (%)
             id               0                   0.0
        company               0                   0.0
      last_name               0                   0.0
     first_name               0                   0.0
  email_address              29                 100.0
      job_title               0                   0.0
 business_phone               0                   0.0
     home_phone              29                 100.0
   mobile_phone              29                 100.0
     fax_number               0                   0.0
        address               0                   0.0
           city     

## Transformacion de datos de cada tabla

### Funciones Propias de Transformación

In [32]:
# Función para limpiar teléfonos
def clean_phone(phone):
    if pd.isna(phone) or phone == '':
        return ''
    return re.sub(r'\D', '', str(phone))

# Función para generar email desde company
def generate_email(company):
    if pd.isna(company) or company == '':
        return ''
    company_clean = re.sub(r'[^a-zA-Z0-9]', '', company.lower())
    return f"{company_clean}@gmail.com"

# Función para limpiar webpage
def clean_webpage(url):
    if pd.isna(url) or url == '':
        return ''
    # Extraer primera URL válida (eliminar # y duplicados)
    urls = re.findall(r'https?://[^\s#]+', str(url))
    return urls[0] if urls else ''

# Función para limpiar product_name y eliminar "Northwind Traders"
def clean_product_name(name):
    if pd.isna(name):
        return ''
    return re.sub(r'^Northwind Traders\s+', '', str(name)).strip()

# Función para convertir supplier_ids de "2;6" a [2, 6]
def parse_supplier_ids(supplier_str):
    if pd.isna(supplier_str) or supplier_str == '':
        return []
    # Convertir "2;6" a [2, 6]
    return [int(x.strip()) for x in str(supplier_str).split(';') if x.strip()]


### Tabla customers

In [26]:
print("🔄 Transformando customers.csv...")
df_customers = pd.read_csv('../data/01_raw/customers.csv')

# Aplicar transformaciones
df_customers['id'] = df_customers['id'].astype(int)

# Limpiar teléfonos (REEMPLAZAR columnas originales, no crear nuevas)
phone_cols = ['business_phone', 'home_phone', 'mobile_phone', 'fax_number']
for col in phone_cols:
    df_customers[col] = df_customers[col].apply(clean_phone) 

# Manejar emails vacíos (REEMPLAZAR columna original)
df_customers['email_address'] = df_customers.apply(
    lambda row: generate_email(row['company']) 
    if pd.isna(row['email_address']) or row['email_address'] == '' 
    else row['email_address'], 
    axis=1
)

# Web page: NaN a string vacío (REEMPLAZAR)
df_customers['web_page'] = df_customers['web_page'].fillna('')

# Eliminar columnas innecesarias
df_customers.drop(columns=['attachments', 'notes'], inplace=True, errors='ignore')

dataframes['customers'] = df_customers
print(f"✅ customers.csv transformado: {len(df_customers)} registros")

🔄 Transformando customers.csv...
✅ customers.csv transformado: 29 registros


### Tabla employees

In [31]:

print("\n🔄 Transformando employees.csv...")
df_employees = pd.read_csv('../data/01_raw/employees.csv')

# Aplicar transformaciones
df_employees['id'] = df_employees['id'].astype(int)

# Limpiar teléfonos (REEMPLAZAR columnas originales)
phone_cols = ['business_phone', 'home_phone', 'mobile_phone', 'fax_number']
for col in phone_cols:
    if col in df_employees.columns:
        df_employees[col] = df_employees[col].apply(clean_phone)  # ← Reemplaza directamente

# Limpiar web_page (REEMPLAZAR columna original)
df_employees['web_page'] = df_employees['web_page'].apply(clean_webpage)

# Email ya está bien, solo asegurar que no hay nulos (REEMPLAZAR)
df_employees['email_address'] = df_employees['email_address'].fillna('')

# Eliminar columnas innecesarias (ahora web_page ya está limpia, no necesitamos eliminarla)
df_employees.drop(columns=['attachments'], inplace=True, errors='ignore')

dataframes['employees'] = df_employees
print(f"✅ employees.csv transformado: {len(df_employees)} registros")


🔄 Transformando employees.csv...
✅ employees.csv transformado: 9 registros


### Tabla products

In [44]:
print("\n🔄 Transformando products.csv...")
df_products = pd.read_csv('../data/01_raw/products.csv')

# Aplicar transformaciones
df_products['id'] = df_products['id'].astype(int)

# Limpiar product_name (REEMPLAZAR columna original)
df_products['product_name'] = df_products['product_name'].apply(clean_product_name)

# Procesar supplier_ids como lista (REEMPLAZAR columna original)
df_products['supplier_ids'] = df_products['supplier_ids'].apply(parse_supplier_ids)
df_products['n_suppliers'] = df_products['supplier_ids'].apply(len)

# Convertir tipos numéricos
df_products['standard_cost'] = pd.to_numeric(df_products['standard_cost'], errors='coerce')
df_products['list_price'] = pd.to_numeric(df_products['list_price'], errors='coerce')
df_products['reorder_level'] = pd.to_numeric(df_products['reorder_level'], errors='coerce').fillna(0).astype(int)
df_products['target_level'] = pd.to_numeric(df_products['target_level'], errors='coerce').fillna(0).astype(int)
df_products['minimum_reorder_quantity'] = pd.to_numeric(df_products['minimum_reorder_quantity'], errors='coerce').fillna(0).astype(int)

# discontinued a booleano
df_products['discontinued'] = df_products['discontinued'].astype(bool)

# Limpiar categoría (REEMPLAZAR)
df_products['category'] = df_products['category'].fillna('').str.strip()

# Eliminar columnas innecesarias (solo attachments, ya que description puede tener info útil)
df_products.drop(columns=['attachments'], inplace=True, errors='ignore')

dataframes['products'] = df_products
print(f"✅ products.csv transformado: {len(df_products)} registros")


🔄 Transformando products.csv...
✅ products.csv transformado: 45 registros


### Tabla orders

In [42]:
print("\n🔄 Transformando orders.csv...")
df_orders = pd.read_csv('../data/01_raw/orders.csv')

# Convertir IDs a enteros (REEMPLAZAR columnas originales)
int_cols = ['id', 'employee_id', 'customer_id', 'shipper_id', 'status_id', 'tax_status_id']
for col in int_cols:
    if col in df_orders.columns:
        df_orders[col] = pd.to_numeric(df_orders[col], errors='coerce').fillna(0).astype(int)

# Convertir fechas (REEMPLAZAR columnas originales)
date_cols = ['order_date', 'shipped_date', 'paid_date']
for col in date_cols:
    if col in df_orders.columns:
        df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce')

# Convertir valores monetarios a float (REEMPLAZAR columnas originales)
float_cols = ['shipping_fee', 'taxes', 'tax_rate']
for col in float_cols:
    if col in df_orders.columns:
        df_orders[col] = pd.to_numeric(df_orders[col], errors='coerce').fillna(0.0)

# Limpiar strings de dirección (REEMPLAZAR columnas originales)
address_cols = ['ship_name', 'ship_address', 'ship_city', 'ship_state_province', 
                'ship_zip_postal_code', 'ship_country_region', 'payment_type']
for col in address_cols:
    if col in df_orders.columns:
        df_orders[col] = df_orders[col].fillna('').astype(str).str.strip()

# Eliminar columnas innecesarias si existen
df_orders.drop(columns=['notes'], inplace=True, errors='ignore')

dataframes['orders'] = df_orders
print(f"✅ orders.csv transformado: {len(df_orders)} registros")


🔄 Transformando orders.csv...
✅ orders.csv transformado: 48 registros


### Tabla order_details

In [40]:
print("\n🔄 Transformando order_details.csv...")
df_order_details = pd.read_csv('../data/01_raw/order_details.csv')

# Convertir IDs a enteros (REEMPLAZAR columnas originales)
int_cols = ['id', 'order_id', 'product_id', 'status_id', 'purchase_order_id', 'inventory_id']
for col in int_cols:
    if col in df_order_details.columns:
        df_order_details[col] = pd.to_numeric(df_order_details[col], errors='coerce').fillna(0).astype(int)

# quantity a entero (está como float con decimales) (REEMPLAZAR)
df_order_details['quantity'] = pd.to_numeric(df_order_details['quantity'], errors='coerce').fillna(0).astype(int)

# unit_price y discount a float (REEMPLAZAR columnas originales)
df_order_details['unit_price'] = pd.to_numeric(df_order_details['unit_price'], errors='coerce').fillna(0.0)
df_order_details['discount'] = pd.to_numeric(df_order_details['discount'], errors='coerce').fillna(0.0)

# date_allocated a datetime (REEMPLAZAR)
df_order_details['date_allocated'] = pd.to_datetime(df_order_details['date_allocated'], errors='coerce')

# Calcular total por línea (NUEVA COLUMNA - no existía antes, está bien crearla)
df_order_details['line_total'] = df_order_details['quantity'] * df_order_details['unit_price'] * (1 - df_order_details['discount'])

dataframes['order_details'] = df_order_details
print(f"✅ order_details.csv transformado: {len(df_order_details)} registros")


🔄 Transformando order_details.csv...
✅ order_details.csv transformado: 58 registros


### Tabla inventory_transactions

In [46]:
print("\n🔄 Transformando inventory_transactions.csv...")
df_inventory = pd.read_csv('../data/01_raw/inventory_transactions.csv')

# Convertir IDs a enteros (REEMPLAZAR columnas originales)
df_inventory['id'] = pd.to_numeric(df_inventory['id'], errors='coerce').astype(int)
df_inventory['transaction_type'] = pd.to_numeric(df_inventory['transaction_type'], errors='coerce').astype(int)
df_inventory['product_id'] = pd.to_numeric(df_inventory['product_id'], errors='coerce').astype(int)
df_inventory['quantity'] = pd.to_numeric(df_inventory['quantity'], errors='coerce').fillna(0).astype(int)

# IDs que pueden ser nulos (REEMPLAZAR columnas originales)
for col in ['purchase_order_id', 'customer_order_id']:
    df_inventory[col] = pd.to_numeric(df_inventory[col], errors='coerce').fillna(0).astype(int)

# Fechas (REEMPLAZAR columnas originales)
df_inventory['transaction_created_date'] = pd.to_datetime(df_inventory['transaction_created_date'], errors='coerce')
df_inventory['transaction_modified_date'] = pd.to_datetime(df_inventory['transaction_modified_date'], errors='coerce')

# Extraer información de comentarios (NUEVAS COLUMNAS - cálculos derivados)
def extract_order_from_comment(comment):
    if pd.isna(comment):
        return 0
    match = re.search(r'Order #(\d+)', str(comment))
    return int(match.group(1)) if match else 0

df_inventory['related_order_id'] = df_inventory['comments'].apply(extract_order_from_comment)
df_inventory['is_backorder'] = df_inventory['comments'].str.contains('Back Ordered', na=False)

# Eliminar columnas innecesarias si existen
df_inventory.drop(columns=['comments'], inplace=True, errors='ignore')

dataframes['inventory_transactions'] = df_inventory


🔄 Transformando inventory_transactions.csv...


### Tabla purchase_orders

In [47]:
print("\n🔄 Transformando purchase_orders.csv...")
df_purchase_orders = pd.read_csv('../data/01_raw/purchase_orders.csv')

# IDs a enteros (REEMPLAZAR columnas originales)
int_cols = ['id', 'supplier_id', 'created_by', 'status_id', 'approved_by', 'submitted_by']
for col in int_cols:
    if col in df_purchase_orders.columns:
        df_purchase_orders[col] = pd.to_numeric(df_purchase_orders[col], errors='coerce').fillna(0).astype(int)

# Fechas (REEMPLAZAR columnas originales)
date_cols = ['submitted_date', 'creation_date', 'payment_date', 'approved_date', 'expected_date']
for col in date_cols:
    if col in df_purchase_orders.columns:
        df_purchase_orders[col] = pd.to_datetime(df_purchase_orders[col], errors='coerce')

# Valores monetarios a float (REEMPLAZAR columnas originales)
float_cols = ['shipping_fee', 'taxes', 'payment_amount']
for col in float_cols:
    if col in df_purchase_orders.columns:
        df_purchase_orders[col] = pd.to_numeric(df_purchase_orders[col], errors='coerce').fillna(0.0)

# Limpiar strings (REEMPLAZAR columnas originales)
df_purchase_orders['payment_method'] = df_purchase_orders['payment_method'].fillna('').astype(str).str.strip()
df_purchase_orders['notes'] = df_purchase_orders['notes'].fillna('').astype(str).str.strip()

dataframes['purchase_orders'] = df_purchase_orders
print(f"✅ purchase_orders.csv transformado: {len(df_purchase_orders)} registros")


🔄 Transformando purchase_orders.csv...
✅ purchase_orders.csv transformado: 28 registros


### Tabla purchase_order_details

In [49]:
print("\n🔄 Transformando purchase_order_details.csv...")
df_po_details = pd.read_csv('../data/01_raw/purchase_order_details.csv')

# IDs a enteros (REEMPLAZAR columnas originales)
int_cols = ['id', 'purchase_order_id', 'product_id', 'inventory_id']
for col in int_cols:
    if col in df_po_details.columns:
        df_po_details[col] = pd.to_numeric(df_po_details[col], errors='coerce').fillna(0).astype(int)

# Cantidades y costos (REEMPLAZAR columnas originales)
df_po_details['quantity'] = pd.to_numeric(df_po_details['quantity'], errors='coerce').fillna(0).astype(int)
df_po_details['unit_cost'] = pd.to_numeric(df_po_details['unit_cost'], errors='coerce').fillna(0.0)

# Fechas y booleanos (REEMPLAZAR columnas originales)
df_po_details['date_received'] = pd.to_datetime(df_po_details['date_received'], errors='coerce')
df_po_details['posted_to_inventory'] = df_po_details['posted_to_inventory'].fillna(0).astype(bool)

# Eliminar columnas innecesarias si existen
df_po_details.drop(columns=['attachments'], inplace=True, errors='ignore')

dataframes['purchase_order_details'] = df_po_details
print(f"✅ purchase_order_details.csv transformado: {len(df_po_details)} registros")


🔄 Transformando purchase_order_details.csv...
✅ purchase_order_details.csv transformado: 55 registros


### Otras tablas

In [54]:
print("\n🔄 Transformando tablas de dimensiones...")

# Lista de tablas pequeñas
small_tables = {
    'order_details_status': ['id', 'status_name'],
    'orders_status': ['id', 'status_name'],
    'orders_tax_status': ['id', 'tax_status_name'],
    'inventory_transaction_types': ['id', 'type_name'],
    'shippers': ['id', 'company', 'last_name', 'first_name', 'job_title'],
    'suppliers': ['id', 'company', 'last_name', 'first_name', 'job_title'],
    'employee_privileges': ['employee_id', 'privilege_id'],
    'privileges': ['id', 'privilege_name'],
    'sales_reports': ['group_by', 'display', 'title', 'filter_row_source', 'default'],
    'strings': ['string_id', 'string_data']
}

for table_name, columns in small_tables.items():
    try:
        df = pd.read_csv(f'../data/01_raw/{table_name}.csv')
        
        # Convertir columnas ID a enteros (REEMPLAZAR columnas originales)
        for col in df.columns:
            if 'id' in col.lower() or col in ['employee_id', 'privilege_id', 'string_id']:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
        
        # Limpiar columnas de texto (REEMPLAZAR columnas originales)
        # Usar exclude=['number'] en lugar de include=['object'] para evitar warning de pandas 3.x
        text_cols = df.select_dtypes(exclude=['number']).columns.tolist()
        for col in text_cols:
            df[col] = df[col].fillna('').astype(str).str.strip()
        
        # Eliminar columnas innecesarias si existen
        df.drop(columns=['attachments', 'web_page', 'email_address', 'business_phone', 
                        'home_phone', 'mobile_phone', 'fax_number', 'address', 'city',
                        'state_province', 'zip_postal_code', 'country_region', 'notes'], 
                inplace=True, errors='ignore')
        
        dataframes[table_name] = df
        print(f"  ✅ {table_name}.csv transformado: {len(df)} registros")
        
    except FileNotFoundError:
        print(f"  ⚠️ {table_name}.csv no encontrado")
    except Exception as e:
        print(f"  ❌ Error en {table_name}: {e}")


🔄 Transformando tablas de dimensiones...
  ✅ order_details_status.csv transformado: 6 registros
  ✅ orders_status.csv transformado: 4 registros
  ✅ orders_tax_status.csv transformado: 2 registros
  ✅ inventory_transaction_types.csv transformado: 4 registros
  ✅ shippers.csv transformado: 3 registros
  ✅ suppliers.csv transformado: 10 registros
  ✅ employee_privileges.csv transformado: 1 registros
  ✅ privileges.csv transformado: 1 registros
  ✅ sales_reports.csv transformado: 5 registros
  ✅ strings.csv transformado: 62 registros


## Guardar datos limpios

In [ ]:
# =============================================================================
# GUARDAR TODAS LAS TABLAS TRANSFORMADAS EN CAPA SILVER
# =============================================================================

print(f"\n💾 Guardando {len(dataframes)} tablas en capa Silver...")

for tabla_nombre, df in dataframes.items():
    try:
        output_file = get_silver_file(tabla_nombre)
        df.to_csv(output_file, index=False, encoding='utf-8')
        
        print(f"  ✅ {tabla_nombre}: {output_file} ({len(df)} registros, {len(df.columns)} columnas)")
        
    except Exception as e:
        print(f"  ❌ Error guardando {tabla_nombre}: {e}")

print(f"\n🎉 Proceso completado. Datos listos en capa SILVER.")


💾 Guardando 18 tablas en capa Silver...
  ✅ customers: D:\proyectos de github\proyecto northwind\data\02_silver\customers.csv (29 registros, 16 columnas)
  ✅ employees: D:\proyectos de github\proyecto northwind\data\02_silver\employees.csv (9 registros, 17 columnas)
  ✅ products: D:\proyectos de github\proyecto northwind\data\02_silver\products.csv (45 registros, 14 columnas)
  ✅ orders: D:\proyectos de github\proyecto northwind\data\02_silver\orders.csv (48 registros, 19 columnas)
  ✅ order_details: D:\proyectos de github\proyecto northwind\data\02_silver\order_details.csv (58 registros, 11 columnas)
  ✅ inventory_transactions: D:\proyectos de github\proyecto northwind\data\02_silver\inventory_transactions.csv (102 registros, 10 columnas)
  ✅ purchase_orders: D:\proyectos de github\proyecto northwind\data\02_silver\purchase_orders.csv (28 registros, 16 columnas)
  ✅ purchase_order_details: D:\proyectos de github\proyecto northwind\data\02_silver\purchase_order_details.csv (55 registr